# MOT17 Dataset Preparation for YOLOv5

This notebook documents the preprocessing pipeline used to prepare the MOT17 dataset for pedestrian detection training using YOLOv5.

The objective is to convert the original MOT17 annotations into YOLO format, organize the dataset into training and validation sets, and verify annotation integrity before training.

## 1. Understanding the MOT17 Dataset

The Multiple Object Tracking 2017 (MOT17) dataset is a benchmark dataset for pedestrian detection and tracking.

Each sequence contains:

- `img1/` : image frames
- `gt/gt.txt` : ground-truth annotations
- `seqinfo.ini` : image metadata

The dataset provides bounding boxes in pixel coordinates.

Example sequence:

MOT17-02-FRCNN

├── img1/

├── gt/

│   └── gt.txt

└── seqinfo.ini

In [ ]:
import os

raw_base = "/content/drive/MyDrive/MOT17/train"

for seq in sorted(os.listdir(raw_base)):
    print(seq)

MOT17-02-DPM
MOT17-02-FRCNN
MOT17-02-SDP
MOT17-04-DPM
MOT17-04-FRCNN
MOT17-04-SDP
MOT17-05-DPM
MOT17-05-FRCNN
MOT17-05-SDP
MOT17-09-DPM
MOT17-09-FRCNN
MOT17-09-SDP
MOT17-10-DPM
MOT17-10-FRCNN
MOT17-10-SDP
MOT17-11-DPM
MOT17-11-FRCNN
MOT17-11-SDP
MOT17-13-DPM
MOT17-13-FRCNN
MOT17-13-SDP


## 2. Exploring MOT17 Annotations

Ground truth labels are stored in `gt.txt`.

Each row represents one object instance in one frame.

Format:

frame_id,
object_id,
x,
y,
width,
height,
confidence,
class_id,
visibility

In [ ]:
gt_path = "/content/drive/MyDrive/MOT17/train/MOT17-02-FRCNN/gt/gt.txt"

with open(gt_path, "r") as f:
    for i in range(5):
        print(f.readline())

1,1,912,484,97,109,0,7,1

2,1,912,484,97,109,0,7,1

3,1,912,484,97,109,0,7,1

4,1,912,484,97,109,0,7,1

5,1,912,484,97,109,0,7,1



## 3. Why Conversion Is Required

MOT17 and yolo have different cordinate systems for the boxes

MO17 has :

x, y, width, height

where (x, y) is the top-left corner.

YOLOv5 requires:

class_id,
x_center,
y_center,
width,
height

where all values are normalized between 0 and 1.

Therefore a conversion pipeline is required.

## 4. MOT17 to YOLO Conversion Pipeline

In [ ]:
import os
import configparser
import shutil


def read_seqinfo(seqinfo_path):
    """Reads seqinfo.ini and returns (width, height) as plain numbers."""
    config = configparser.ConfigParser()
    config.read(seqinfo_path)
    width = int(config['Sequence']['imWidth'])
    height = int(config['Sequence']['imHeight'])
    return width, height


def convert_sequence(sequence_folder, output_images_folder, output_labels_folder):
    """
    sequence_folder: path to one sequence, e.g. .../MOT17-02-FRCNN
    output_images_folder: where to COPY the images to
    output_labels_folder: where to WRITE the new YOLO-format label files
    """
    seqinfo_path = os.path.join(sequence_folder, 'seqinfo.ini')
    img_width, img_height = read_seqinfo(seqinfo_path)

    gt_path = os.path.join(sequence_folder, 'gt', 'gt.txt')
    img_dir = os.path.join(sequence_folder, 'img1')

    # Step A: read every line of gt.txt and group boxes by which frame they belong to
    boxes_by_frame = {}
    with open(gt_path, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            frame_id = int(parts[0])
            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])
            is_valid = int(parts[6])   # 1 = keep, 0 = skip
            class_id = int(parts[7])   # 1 = pedestrian

            if is_valid != 1 or class_id != 1:
                continue  # skip junk boxes and non-pedestrian boxes

            boxes_by_frame.setdefault(frame_id, []).append((x, y, w, h))

    # Step B: make the output folders if they don't exist yet
    os.makedirs(output_images_folder, exist_ok=True)
    os.makedirs(output_labels_folder, exist_ok=True)

    # Step C: go through every image, copy it across, and write its matching label file
    image_files = sorted(os.listdir(img_dir))
    for image_name in image_files:
        frame_id = int(os.path.splitext(image_name)[0])  # "000001.jpg" -> 1

        shutil.copy(os.path.join(img_dir, image_name),
                    os.path.join(output_images_folder, image_name))

        label_name = os.path.splitext(image_name)[0] + '.txt'
        label_path = os.path.join(output_labels_folder, label_name)

        lines_to_write = []
        for (x, y, w, h) in boxes_by_frame.get(frame_id, []):
            x_center = (x + w / 2) / img_width
            y_center = (y + h / 2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height
            lines_to_write.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        # even if a frame has zero people in it, we still write an empty file —
        # YOLO expects every image to have a label file, even an empty one
        with open(label_path, 'w') as f:
            f.write('\n'.join(lines_to_write))

    print(f"Done: {sequence_folder} -> {len(image_files)} images processed")

### Bounding Box Validation

During preprocessing it was discovered that some MOT17 annotations extend beyond image boundaries.

This caused YOLOv5 to reject labels as corrupt.

To solve this issue, bounding boxes were clipped to image dimensions before normalization.

##New conversion function to account for extending boxes

In [ ]:
import os
import configparser
import shutil


def read_seqinfo(seqinfo_path):
    config = configparser.ConfigParser()
    config.read(seqinfo_path)
    width = int(config['Sequence']['imWidth'])
    height = int(config['Sequence']['imHeight'])
    return width, height


def convert_sequence(sequence_folder, output_images_folder, output_labels_folder):
    seqinfo_path = os.path.join(sequence_folder, 'seqinfo.ini')
    img_width, img_height = read_seqinfo(seqinfo_path)

    gt_path = os.path.join(sequence_folder, 'gt', 'gt.txt')
    img_dir = os.path.join(sequence_folder, 'img1')

    boxes_by_frame = {}
    with open(gt_path, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            frame_id = int(parts[0])
            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])
            is_valid = int(parts[6])
            class_id = int(parts[7])

            if is_valid != 1 or class_id != 1:
                continue

            # NEW: clamp the box so it never extends past the image edges
            x = max(0, x)
            y = max(0, y)
            w = min(w, img_width - x)
            h = min(h, img_height - y)

            boxes_by_frame.setdefault(frame_id, []).append((x, y, w, h))

    os.makedirs(output_images_folder, exist_ok=True)
    os.makedirs(output_labels_folder, exist_ok=True)

    image_files = sorted(os.listdir(img_dir))
    for image_name in image_files:
        frame_id = int(os.path.splitext(image_name)[0])

        shutil.copy(os.path.join(img_dir, image_name),
                    os.path.join(output_images_folder, image_name))

        label_name = os.path.splitext(image_name)[0] + '.txt'
        label_path = os.path.join(output_labels_folder, label_name)

        lines_to_write = []
        for (x, y, w, h) in boxes_by_frame.get(frame_id, []):
            x_center = (x + w / 2) / img_width
            y_center = (y + h / 2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height
            lines_to_write.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        with open(label_path, 'w') as f:
            f.write('\n'.join(lines_to_write))

    print(f"Done: {sequence_folder} -> {len(image_files)} images processed")

In [1]:
train_sequences = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN"
]

val_sequences = [
    "MOT17-11-FRCNN"
]

print("Train:", train_sequences)
print("Validation:", val_sequences)

Train: ['MOT17-02-FRCNN', 'MOT17-04-FRCNN', 'MOT17-05-FRCNN', 'MOT17-09-FRCNN', 'MOT17-10-FRCNN']
Validation: ['MOT17-11-FRCNN']


## 5. Dataset Configuration

After conversion, the dataset was organized into:

MOT17_processed/

├── train/

│   ├── images/

│   └── labels/

├── val/

│   ├── images/

│   └── labels/

└── data.yaml

In [2]:
data_yaml = """
train: /content/drive/MyDrive/MOT17_processed/train/images

val: /content/drive/MyDrive/MOT17_processed/val/images

nc: 1

names: ['person']
"""

print(data_yaml)


train: /content/drive/MyDrive/MOT17_processed/train/images

val: /content/drive/MyDrive/MOT17_processed/val/images

nc: 1

names: ['person']



## 6. Summary

The MOT17 dataset was successfully converted into YOLOv5 format.

Key tasks completed:

- Annotation format conversion
- Train-validation split creation
- Label validation
- Bounding box clipping
- Dataset configuration generation

The processed dataset was subsequently used to train three YOLOv5 variants for pedestrian detection.